In [1]:
!pip install -qU FlagEmbedding pythainlp rank-bm25 pandas requests python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires pandas<2.4.0,>=2.0.0, but you have pandas 3.0.1 which is incompatible.
gluonts 0.16.2 requires pandas<3,>=1.0, but you have pandas 3.0.1 which is incompatible.
autogluon-core 1.5.0 requires pandas<2.4.0,>=2.0.0, but you have pandas 3.0.1 which is incompatible.
autogluon-features 1.5.0 requires pandas<2.4.0,>=2.0.0, but you have pandas 3.0.1 which is incompatible.
autogluon-tabular 1.5.0 requires pandas<2.4.0,>=2.0.0, but you have pandas 3.0.1 which is incompatible.
autogluon-common 1.5.0 requires pandas<2.4.0,>=2.0.0, but you have pandas 3.0.1 which is incompatible.
autogluon-timeseries 1.5.0 requires pandas<2.4.0,>=2.0.0, but you have pandas 3.0.1 which is incompatible.


## การตั้งค่า Environment & Imports
กำหนดค่าตัวแปรเบื้องต้นและโหลด Keys สำหรับการเรียก API ของ ThaiLLM

In [11]:
import os
import re
import csv
import time
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from pythainlp.tokenize import word_tokenize
from FlagEmbedding import BGEM3FlagModel, FlagReranker

THAILLM_API_KEY = "zQfWsOmMlpsfGuXkMkBrMZW5CsAJBEp1"

N_QUESTIONS = 100
DATA_DIR = "/home/drasogun/DraSoGun/AI/SuperAI_ss6/Competitions/FahMai-RAG/data"
KB_DIR = f"{DATA_DIR}/knowledge_base"

## Semantic Markdown Chunking & Metadata Injection
คลาสสำหรับทำการอ่านเอกสาร Markdown แทนการทำ Sliding Window จะเปลี่ยนเป็นการตัดด้วยโครงสร้าง Markdown (Header-aware) พร้อมฝัง Metadata อัตโนมัติลงไปในทุก Chunk

In [3]:
class AdvancedChunker:
    def __init__(self, kb_path):
        self.kb_path = Path(kb_path)
        
    def process(self):
        chunks = []
        if not self.kb_path.exists():
            return chunks
            
        for fp in self.kb_path.rglob("*.md"):
            text = fp.read_text(encoding="utf-8")
            category = str(fp.parent.name)
            doc_name = fp.stem
            
            sections = re.split(r'\n(?=#+ )', text) 
            for sec in sections:
                if len(sec.strip()) < 10:
                    continue
                enriched_chunk = f"[Document: {doc_name} | Category: {category}]\n{sec.strip()}"
                chunks.append({
                    "text": enriched_chunk,
                    "source": str(fp.relative_to(self.kb_path)),
                    "category": category
                })
        return chunks

chunker = AdvancedChunker(KB_DIR)
chunks = chunker.process()

## การเตรียม Embedding ทรงพลัง (BGE-M3) และ Sparse (BM25)
นำ Chunk เข้าสู่กระบวนการสร้าง Vector ด้วยบัฟเฟอร์ขนาดใหญ่แบบ Dense Vectors และประมวลผล Sparse Lexical Scoring คู่ขนานกันไป

In [4]:
if chunks:
    m3_model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
    chunk_texts = [c["text"] for c in chunks]
    
    embeddings_out = m3_model.encode(chunk_texts, batch_size=12, max_length=512)
    dense_embeddings = embeddings_out['dense_vecs']
    
    from rank_bm25 import BM25Okapi
    tokenized_chunks = [word_tokenize(c["text"], engine="newmm") for c in chunks]
    bm25 = BM25Okapi(tokenized_chunks)

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

...

onnx/model.onnx_data:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

pre tokenize: 100%|██████████| 74/74 [00:00<00:00, 255.93it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 74/74 [00:07<00:00,  9.49it/s]


## กองกำลังรบพิเศษ: Cross-Encoder Reranker
โหลดโมเดล Reranker สำหรับจัดเรียงเอกสารที่ถูกคัดกรองเบื้องต้นขึ้นมาใหม่ด้วยสถิติความแม่นยำสูงสุด

In [5]:
if chunks:
    reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

## ห่วงโซ่การค้นหาแบบ Hybrid & Precision RRF
ฟังก์ชันการทำ Retrieval แบบ 3 ขั้นตอน: Dense -> Sparse -> RRF Fusion -> Cross-Encoder

In [6]:
def shadow_retrieve(query, top_k_fusion=30, final_k=5):
    q_out = m3_model.encode([query], max_length=512)['dense_vecs']
    scores_dense = np.dot(dense_embeddings, q_out.T).flatten()
    dense_idx = np.argsort(scores_dense)[::-1][:top_k_fusion]
    
    tokens = word_tokenize(query, engine="newmm")
    scores_bm25 = bm25.get_scores(tokens)
    bm25_idx = np.argsort(scores_bm25)[::-1][:top_k_fusion]
    
    rrf_k = 60
    rrf_scores = {}
    for rank, idx in enumerate(dense_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(bm25_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)
        
    fused_idx = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:top_k_fusion]
    fused_chunks = [chunks[i]["text"] for i in fused_idx]
    
    pairs = [[query, text] for text in fused_chunks]
    rerank_scores = reranker.compute_score(pairs)
    
    best_relative_idx = np.argsort(rerank_scores)[::-1][:final_k]
    final_docs = [fused_chunks[i] for i in best_relative_idx]
    
    return final_docs

## Prompt Engineering เชิงแทคติก (Chain-of-Thought)
การออกแบบ Prompt สำหรับ ThaiLLM เพื่อลดทอนการเกิด Hallucination บังคับให้คิดทีละสเต็ป

In [7]:
SYSTEM_PROMPT = """คุณคือผู้เชี่ยวชาญด้านข้อมูลของร้านอุปกรณ์อิเล็กทรอนิกส์ "ฟ้าใหม่"
งานของคุณคือตอบคำถามแบบตัวเลือกปรนัย โดยใช้ข้อมูลจาก Context ที่ให้มาเท่านั้น
กฎเหล็ก:
- วิเคราะห์ข้อมูลทีละขั้นตอน (Step-by-step thinking)
- หากข้อมูลใน Context อธิบายได้ตรงกับชอยส์ 1-8 ให้ตอบชอยส์นั้น
- หาก Context ไม่มีเนื้อหาที่สามารถตอบคำถามนี้พิจารณาได้เลย ให้ตอบตัวเลือก "9. ไม่มีข้อมูลนี้ในฐานข้อมูล"
- หากคำถามไม่เกี่ยวกับเรื่องร้านฟ้าใหม่ สินค้า หรือบริการเลยแม้แต่น้อย ให้ตอบ "10. คำถามนี้ไม่เกี่ยวข้องกับร้านฟ้าใหม่"
- ในบรรทัดสุดท้าย คุณต้องตอบในรูปแบบ "ANSWER: X" (X คือตัวเลข 1 ถึง 10 เท่านั้น)"""

def build_advanced_prompt(question, choices, contexts):
    ctx_str = "\n\n".join([f"--- Context {i+1} ---\n{c}" for i, c in enumerate(contexts)])
    choices_str = "\n".join(f"{k}. {v}" for k, v in choices.items())
    return f"""บริบทข้อมูล (Context):
{ctx_str}

คำถาม: {question}

ตัวเลือก:
{choices_str}

จงวิเคราะห์ความเชื่อมโยงก่อน จากนั้นสรุปโดยพิมพ์ ANSWER: X ด้านล่างสุด"""

def parse_cot_answer(text):
    if not text: return 1
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    m = re.search(r"ANSWER:\s*(\d+)", clean)
    if m:
        val = int(m.group(1))
        return val if 1 <= val <= 10 else 1
    for d in re.findall(r"\b(\d{1,2})\b", text):
        if 1 <= int(d) <= 10: return int(d)
    return 1

## ThaiLLM Endpoint & Fallback Connection
ฟังก์ชันยิง API ไปยัง ThaiLLM ป้องกัน Rate Limit ด้วย Exponential Backoff

In [12]:
def ask_llm(messages, model="openthaigpt", max_retries=5):
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2024,
        "temperature": 0,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)

            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()

        except requests.exceptions.RequestException as e:
            wait = 2 ** attempt
            time.sleep(wait)

    return None

## The Global Execution Pipeline
การรันข้อมูลทั้งหมดจาก `questions.csv` ทำการควบรวม ส่งวิเคราะห์ คัดกรองคำตอบ และประมวลผลเป็น `submission.csv` ขั้นสุดท้าย

In [13]:
from tqdm.auto import tqdm
import csv
import time

questions = []
try:
    with open(f"{DATA_DIR}/questions.csv", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
            questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})
except FileNotFoundError:
    pass

predictions = {}

if chunks:
    print(f"[+] Commencing Global Execution on {len(questions[:N_QUESTIONS])} targets...")
    
    for q in tqdm(questions[:N_QUESTIONS], desc="Neutralizing Targets", unit="Q"):
        top_contexts = shadow_retrieve(q["question"], top_k_fusion=30, final_k=4)
        prompt = build_advanced_prompt(q["question"], q["choices"], top_contexts)
        
        raw = ask_llm([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ], model="typhoon") 
        
        pred = parse_cot_answer(raw)
        predictions[q["id"]] = pred
        time.sleep(0.5)

    with open("submission_darkforge.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "answer"])
        for q in questions[:N_QUESTIONS]:
            writer.writerow([q["id"], predictions.get(q["id"], 1)])
            
    print("[!] Mission Accomplished. Weaponized submission generated.")

[+] Commencing Global Execution on 100 targets...


Neutralizing Targets:   0%|          | 0/100 [00:00<?, ?Q/s]

[!] Mission Accomplished. Weaponized submission generated.
